In [2]:
import requests as req
import pandas as pd
import datetime as dt
import time

# Настройки
LATITUDE = 54.10
LONGITUDE = 52.66
START_DATE = pd.to_datetime("2023-12-31")
END_DATE = pd.to_datetime("2025-06-16")
# Используем URL архива (для дат до "сегодня - 2-5 дней")
URL = "https://archive-api.open-meteo.com/v1/archive"
SIGNS = [
    # Температура (2 метра)
    "temperature_2m_max", "temperature_2m_min", "temperature_2m_mean",
    "apparent_temperature_max", "apparent_temperature_min", "apparent_temperature_mean",
    
    # Осадки
    "precipitation_sum", "rain_sum", "snowfall_sum", "precipitation_hours",
    
    # Солнце и радиация
    "sunrise", "sunset", "daylight_duration", "sunshine_duration",
    "shortwave_radiation_sum", "uv_index_max",
    
    # Ветер (10 метров)
    "wind_speed_10m_max", "wind_gusts_10m_max", "wind_direction_10m_dominant",
    
    # Почва (0-7 см)
    "soil_temperature_0_to_7cm_mean", "soil_moisture_0_to_7cm_mean",
    
    # Прочее
    "et0_fao_evapotranspiration", "weather_code"
]

def get_chunk_per_date(start_date, end_date):
    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": start_date,
        "end_date": end_date,
        "daily": ",".join(SIGNS),  # Ключевое отличие: daily вместо hourly
        "timezone": "auto"
    }   

    try:
        response = req.get(URL, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()

        # Проверка наличия данных в ответе
        if 'daily' not in data:
            print(f"Нет данных за период {start_date} - {end_date}")
            return None

        # Сборка DataFrame из секции 'daily'
        dataframe = pd.DataFrame({'date': pd.to_datetime(data['daily']['time'])})
        for param in SIGNS:
            dataframe[param] = data['daily'][param]

        return dataframe
    except Exception as e:
        print(f"Ошибка при запросе {start_date}: {e}")
        return None

def load_data(chunk_days=60):
    all_data = []
    current = START_DATE
    
    while current <= END_DATE:
        # Рассчитываем конец текущего чанка
        chunk_end = min(current + dt.timedelta(days=chunk_days - 1), END_DATE)
        
        print(f"Загрузка чанка: {current.date()} >>> {chunk_end.date()}")
        
        chunk_df = get_chunk_per_date(
            current.strftime('%Y-%m-%d'), 
            chunk_end.strftime('%Y-%m-%d')
        )
    
        if chunk_df is not None:
            all_data.append(chunk_df)
    
        current = chunk_end + dt.timedelta(days=1)
        time.sleep(0.5) # Пауза для соблюдения лимитов API

    if not all_data:
        return pd.DataFrame()
        
    return pd.concat(all_data, ignore_index=True)

# Запуск процесса
weather_data = load_data(chunk_days=90)
weather_data = pd.DataFrame(weather_data)
weather_data.to_excel("hist_weather.xlsx")

Загрузка чанка: 2023-12-31 >>> 2024-03-29
Загрузка чанка: 2024-03-30 >>> 2024-06-27
Загрузка чанка: 2024-06-28 >>> 2024-09-25
Загрузка чанка: 2024-09-26 >>> 2024-12-24
Загрузка чанка: 2024-12-25 >>> 2025-03-24
Загрузка чанка: 2025-03-25 >>> 2025-06-16
